<a href="https://colab.research.google.com/github/roly-3618/114-2_coding/blob/main/%E3%80%8CHW2_%E6%88%90%E7%B8%BE%E4%B8%80%E6%9C%AC%E9%80%9A_ipynb%E3%80%8D%E7%9A%84%E5%89%AF%E6%9C%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q google-generativeai

In [ ]:
import gradio as gr
import pandas as pd
from google.colab import auth
from google.auth import default

# -*- coding: utf-8 -*-
import gspread
from datetime import datetime
import google.generativeai as genai
import os
import json

In [ ]:
from google.colab import userdata

# 從 Colab Secrets 中獲取 API 金鑰
api_key = userdata.get('gemini')

# 使用獲取的金鑰配置 genai
genai.configure(api_key=api_key)

model = genai.GenerativeModel('gemini-2.5-flash')

In [ ]:
SHEET_URL = "https://docs.google.com/spreadsheets/d/1f0jkB_z5XxHhVhpB2PVDENNQ_PUrb-OSHE7CGpJg6G8/edit?usp=sharing"
WORKSHEET_NAME = "工作表2"

REQUIRED_COLUMNS = ["日期", "科目", "作業成績"]

_auth_done = False
_gc = None
_ws = None

In [ ]:
# --- 主要功能區塊 ---
def get_user_grades():
    """
    透過終端機輸入學生成績，直到使用者輸入 'q' 結束。
    """
    print("--- 準備輸入成績。輸入 'q' 來停止。---")
    grades = []
    while True:
        subject = input("請輸入科目（或輸入 'q' 停止）：")
        if subject.lower() == 'q':
            break

        grade = input(f"請輸入 {subject} 的成績：")
        try:
            grade = int(grade)
        except ValueError:
            print("成績必須是數字。請重新輸入。")
            continue

        today = datetime.now().strftime('%Y-%m-%d')
        grades.append([today, subject, grade])
        print(f"已記錄：日期: {today}, 科目: {subject}, 成績: {grade}\n")

    return grades

In [ ]:
def get_ai_summary(grades):
    """
    呼叫 Gemini 模型來生成成績摘要與常見迷思。
    """
    # 準備給 AI 的提示
    prompt_text = "以下是學生的成績列表，請幫我根據這些成績，產出一個簡單的摘要與常見迷思整理（不評分，只做總結）。\n\n"
    for record in grades:
        date, subject, grade = record
        prompt_text += f"日期：{date}, 科目：{subject}, 成績：{grade}\n"

    print("\n--- 正在呼叫 AI 模型生成摘要... ---")
    try:
        response = client.models.generate_content(model = MODEL_ID, contents = prompt_text)
        summary = response.text
        return summary
    except Exception as e:
        print(f"呼叫 AI 時發生錯誤：{e}")
        return "AI 摘要生成失敗。"

In [ ]:

new_grades = get_user_grades()

--- 準備輸入成績。輸入 'q' 來停止。---
請輸入科目（或輸入 'q' 停止）：國文
請輸入 國文 的成績：88
已記錄：日期: 2026-04-27, 科目: 國文, 成績: 88

請輸入科目（或輸入 'q' 停止）：數學
請輸入 數學 的成績：77
已記錄：日期: 2026-04-27, 科目: 數學, 成績: 77

請輸入科目（或輸入 'q' 停止）：英文
請輸入 英文 的成績：66
已記錄：日期: 2026-04-27, 科目: 英文, 成績: 66

請輸入科目（或輸入 'q' 停止）：q


In [ ]:
get_ai_summary(new_grades)


--- 正在呼叫 AI 模型生成摘要... ---


'好的，這是一份基於您提供的成績列表所做的簡單摘要與常見迷思整理，全程不對學生的表現進行評分或判斷：\n\n---\n\n### **學生學業成績摘要**\n\n以下為學生在特定日期各科目的成績概述：\n\n*   **評量日期：** 2026年4月27日\n*   **評量科目及成績：**\n    *   國文：88分\n    *   數學：77分\n    *   英文：66分\n*   **成績範圍：** 這次評量的科目成績介於66分至88分之間。\n*   **科目表現：** 國文科目得分最高，英文科目得分較低。\n\n---\n\n### **關於學生學業成績的常見迷思整理**\n\n成績單提供了一部分學習成果的資訊，但對於其解讀，社會上常有一些普遍存在的誤解。以下將整理一些常見迷思，以助於更全面地看待學業成績：\n\n1.  **迷思一：分數高低是衡量學生全部能力與價值的唯一標準。**\n    *   **事實：** 成績單主要反映的是學生在特定時間點、特定科目上對知識或技能的掌握程度。它無法全面衡量學生的創造力、解決問題能力、情商、人際溝通、實踐操作能力或個人品格等多元面向。這些能力對於學生的成長與未來發展同樣重要。\n\n2.  **迷思二：低分就代表學生不努力或不聰明。**\n    *   **事實：** 影響成績的因素非常多，包括學習方法、考試當天的身體狀況、考試焦慮、教材難度、老師的教學方式、科目本身的興趣程度，甚至是學習風格與評量方式不符等。分數低不代表不努力或不聰明，可能只是需要調整學習策略、克服特定困難或發現更適合自己的學習方式。\n\n3.  **迷思三：學業成績能完全預測未來的成功。**\n    *   **事實：** 學業成績與未來的學術或職業發展有一定關聯性，但這並非絕對。現實生活中，許多其他關鍵技能，如適應力、創新思維、團隊合作、溝通能力和恆毅力，對個人成功扮演著同等甚至更重要的角色。許多在學業上表現平平的人，也能在特定領域取得卓越成就。\n\n4.  **迷思四：直接比較不同學生的成績是最佳的激勵方式。**\n    *   **事實：** 每個學生的學習進度、背景和優勢都不同。過度強調與他人的分數比較，可能導致不必要的壓力和焦慮，甚至扼殺學習興趣。更有效的做法是關注學生自身的進步和成長，鼓勵他們超越自己，並找出適合

In [ ]:
def main():
    """
    主程式流程：輸入成績 -> 獲取 AI 摘要 -> 寫入 Google Sheet。
    """
    try:
        # 1. Google Sheet 身份驗證
        auth.authenticate_user()

        creds, _ = default()
        gc = gspread.authorize(creds)

        sh = gc.open_by_url(SHEET_URL)
        ws = sh.worksheet(WORKSHEET_NAME)





        print("--- Google Sheet 連線成功。---")

        # 2. 獲取使用者輸入的成績
        new_grades = get_user_grades()

        if not new_grades:
            print("沒有輸入任何成績，程式結束。")
            return

        # 3. 將新成績寫入 Google Sheet
        ws.append_rows(new_grades)
        print("\n--- 成績已成功寫入 Google Sheet。---")

        # 4. 獲取 AI 摘要並寫入 Google Sheet
        summary = get_ai_summary(new_grades)

        # 尋找第一行空白列
        next_row = len(ws.col_values(1)) + 1

        # 使用 update_cell() 方法逐一更新儲存格
        ws.update_cell(next_row, 1, datetime.now().strftime('%Y-%m-%d'))
        ws.update_cell(next_row, 2, 'AI 摘要')

        # 為了避免單元格內容過長，將摘要內容分成多行來寫入
        summary_lines = summary.split('\n')
        for i, line in enumerate(summary_lines):
            ws.update_cell(next_row + i, 3, line)

        print("\n--- AI 摘要已成功寫入 Google Sheet。---")
        print("以下是 AI 生成的摘要內容：")
        print("-" * 50)
        print(summary)
        print("-" * 50)

    except gspread.exceptions.APIError as e:
        print(f"Google Sheets API 錯誤：{e.response.text}")
        print("請確認：")
        print("1. 您的服務帳戶金鑰檔案正確且未過期。")
        print("2. 您已將服務帳戶的 Email 地址（在 JSON 檔案中）分享給 Google Sheet，並給予編輯權限。")
    except Exception as e:
        print(f"發生未預期的錯誤：{e}")

if __name__ == "__main__":
    main()

--- Google Sheet 連線成功。---
--- 準備輸入成績。輸入 'q' 來停止。---
請輸入科目（或輸入 'q' 停止）：國文
請輸入 國文 的成績：88
已記錄：日期: 2026-04-27, 科目: 國文, 成績: 88

請輸入科目（或輸入 'q' 停止）：數學
請輸入 數學 的成績：77
已記錄：日期: 2026-04-27, 科目: 數學, 成績: 77

請輸入科目（或輸入 'q' 停止）：英文
請輸入 英文 的成績：66
已記錄：日期: 2026-04-27, 科目: 英文, 成績: 66

請輸入科目（或輸入 'q' 停止）：q

--- 成績已成功寫入 Google Sheet。---

--- 正在呼叫 AI 模型生成摘要... ---

--- AI 摘要已成功寫入 Google Sheet。---
以下是 AI 生成的摘要內容：
--------------------------------------------------
好的，這是根據您提供的學生單次成績，所整理的簡單摘要與常見迷思：

---

### **學生單次成績摘要**

這份成績單展示了學生在三個科目上的表現。
*   **國文科目：** 獲得88分。
*   **數學科目：** 獲得77分。
*   **英文科目：** 獲得66分。

整體而言，這次測驗中，國文科目的分數相對較高，英文科目的分數則相對較低，而數學科目的分數介於兩者之間。

---

### **常見迷思整理（不評分）**

在解讀學生成績時，我們常會有一些普遍的觀念，以下列出幾個常見迷思，以中立角度說明：

1.  **迷思一：單一分數能完全代表學生能力。**
    *   **澄清：** 學生成績是衡量學習成果的其中一種方式，而非能力的唯一或絕對指標。考試當天的狀態（如精神、壓力）、題目難易度、甚至出題方向，都可能影響最終分數。它更像是一個「學習的當下快照」，提供一個觀察點。

2.  **迷思二：不同科目的分數可以直接比較優劣。**
    *   **澄清：** 不同科目有其獨特的學習內容、評量標準與難易度。即使分數有高低差異，也不代表在某科目表現較不努力或能力不足，可能只是該科目學習曲線或學習策略尚待調整。例如，一個科目的88分與另一個科目的66分，可能因為其學習性質不同，而